# 02 · Local Unfolding and Random Controls

This notebook refines the spacing analysis by comparing three views of nontrivial zeta-zero spacings:

1. **raw gaps**
2. **global normalization** by one mean gap
3. **local unfolding / local normalization** using a moving average

Then it compares the zeta data against simple randomized controls.

## Why this matters

For the Riemann zeta zeros, the average spacing changes with height.
So a single global mean is useful for a first pass, but it mixes early wider gaps with later tighter gaps.

This notebook stays in a **VC-style** mode:

- standard zeta-zero data from `mpmath`
- explicit normalization choices
- randomized controls stated clearly
- no theorem-level claims from small-sample numerics

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50

## Collect nontrivial zeros and raw gaps

In [ ]:
N = 200
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
gaps = np.diff(t)

len(t), len(gaps), t[:5], gaps[:5]

## Global normalization

Define

\[
g_n^{(\mathrm{global})} = \frac{g_n}{\langle g \rangle}.
\]

This is simple, but it does not account for the slow height dependence of the mean gap.

In [ ]:
global_mean_gap = gaps.mean()
global_norm_gaps = gaps / global_mean_gap

global_mean_gap

## Local unfolding via moving average

A lightweight local normalization is:

\[
g_n^{(\mathrm{local})} = \frac{g_n}{\text{local mean around } g_n}.
\]

This is not a full analytic unfolding procedure, but it is a transparent local correction.

In [ ]:
window = 11  # odd window keeps the center interpretation simple

kernel = np.ones(window) / window
local_means = np.convolve(gaps, kernel, mode="same")

# Fix edge effects by replacing the first/last few entries with smaller-window means
half = window // 2
for i in range(half):
    local_means[i] = gaps[:i + half + 1].mean()
    local_means[-(i + 1)] = gaps[-(i + half + 1):].mean()

local_norm_gaps = gaps / local_means

local_means[:5], local_norm_gaps[:5]

## Compare raw, global-normalized, and local-normalized gaps

In [ ]:
gap_index = np.arange(1, len(gaps) + 1)

plt.figure(figsize=(8, 4.8))
plt.plot(gap_index, gaps, linewidth=1, marker="o", label="raw gaps")
plt.axhline(global_mean_gap, linestyle="--", linewidth=1, label="global mean")
plt.xlabel("gap index n")
plt.ylabel(r"$g_n$")
plt.title("Raw zeta-zero spacings")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4.8))
plt.plot(gap_index, global_norm_gaps, linewidth=1, marker="o", label="global norm")
plt.axhline(1.0, linestyle="--", linewidth=1)
plt.xlabel("gap index n")
plt.ylabel(r"$g_n / \langle g \rangle$")
plt.title("Global normalization of zeta-zero spacings")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4.8))
plt.plot(gap_index, local_norm_gaps, linewidth=1, marker="o", label="local norm")
plt.axhline(1.0, linestyle="--", linewidth=1)
plt.xlabel("gap index n")
plt.ylabel("locally normalized gap")
plt.title("Local unfolding / local normalization of zeta-zero spacings")
plt.show()

## Histograms: global vs local normalization

In [ ]:
plt.figure(figsize=(8, 4.8))
plt.hist(global_norm_gaps, bins=22, alpha=0.7, label="global normalization")
plt.hist(local_norm_gaps, bins=22, alpha=0.7, label="local normalization")
plt.xlabel("normalized spacing")
plt.ylabel("count")
plt.title("Zeta-zero spacing histograms")
plt.legend()
plt.show()

## Randomized controls

We compare zeta-zero local-normalized spacings against two simple controls:

1. **shuffled gaps**: same gap values, random order  
2. **exponential control**: a Poisson-like spacing baseline with mean 1

These controls do not reproduce the full mathematics of zeta zeros.
They are just useful contrast classes.

In [ ]:
rng = np.random.default_rng(9423)

shuffled_local = rng.permutation(local_norm_gaps)
exp_control = rng.exponential(scale=1.0, size=len(local_norm_gaps))

## Histogram comparison with controls

In [ ]:
plt.figure(figsize=(8.5, 5.0))
bins = np.linspace(0, max(local_norm_gaps.max(), exp_control.max()), 24)
plt.hist(local_norm_gaps, bins=bins, alpha=0.6, density=True, label="zeta local norm")
plt.hist(shuffled_local, bins=bins, alpha=0.4, density=True, label="shuffled local")
plt.hist(exp_control, bins=bins, alpha=0.4, density=True, label="exponential control")
plt.xlabel("normalized spacing")
plt.ylabel("density")
plt.title("Local-normalized zeta spacings vs controls")
plt.legend()
plt.show()

## Local triplet balance after local normalization

Using locally normalized gaps, define for each center zero:

\[
b_n = \frac{\min(g_L,g_R)}{\max(g_L,g_R)}.
\]

We compare the zeta sequence against the same metric applied to controls.

In [ ]:
zeta_left = local_norm_gaps[:-1]
zeta_right = local_norm_gaps[1:]
zeta_balance = np.minimum(zeta_left, zeta_right) / np.maximum(zeta_left, zeta_right)

shuf_left = shuffled_local[:-1]
shuf_right = shuffled_local[1:]
shuf_balance = np.minimum(shuf_left, shuf_right) / np.maximum(shuf_left, shuf_right)

exp_left = exp_control[:-1]
exp_right = exp_control[1:]
exp_balance = np.minimum(exp_left, exp_right) / np.maximum(exp_left, exp_right)

zeta_balance.mean(), shuf_balance.mean(), exp_balance.mean()

In [ ]:
plt.figure(figsize=(8.5, 5.0))
bins = np.linspace(0, 1, 22)
plt.hist(zeta_balance, bins=bins, alpha=0.6, density=True, label="zeta balance")
plt.hist(shuf_balance, bins=bins, alpha=0.5, density=True, label="shuffled balance")
plt.hist(exp_balance, bins=bins, alpha=0.5, density=True, label="exponential balance")
plt.xlabel(r"$b_n = \min(g_L,g_R)/\max(g_L,g_R)$")
plt.ylabel("density")
plt.title("Triplet-balance comparison")
plt.legend()
plt.show()

## Running local-balance series

Plot the local-balance series itself for the zeta data.

In [ ]:
center_index = np.arange(2, len(t))

plt.figure(figsize=(8, 4.8))
plt.plot(center_index, zeta_balance, linewidth=1, marker="o")
plt.axhline(zeta_balance.mean(), linestyle="--", linewidth=1, label=f"mean ≈ {zeta_balance.mean():.3f}")
plt.xlabel("center zero index n")
plt.ylabel("local balance")
plt.title("Zeta local triplet balance after local normalization")
plt.ylim(0, 1.05)
plt.legend()
plt.show()

## Simple summary statistics

In [ ]:
summary = {
    "raw_gap_mean": float(gaps.mean()),
    "raw_gap_std": float(gaps.std()),
    "global_norm_mean": float(global_norm_gaps.mean()),
    "global_norm_std": float(global_norm_gaps.std()),
    "local_norm_mean": float(local_norm_gaps.mean()),
    "local_norm_std": float(local_norm_gaps.std()),
    "zeta_balance_mean": float(zeta_balance.mean()),
    "shuffled_balance_mean": float(shuf_balance.mean()),
    "exp_balance_mean": float(exp_balance.mean()),
}
summary

## Notes

- Global normalization is a useful first pass, but local normalization is better matched to the changing average spacing with height.
- The shuffled control preserves the same set of gap values but destroys local order.
- The exponential control is a simple Poisson-like baseline.
- These comparisons are exploratory, but they start to test whether local descriptors distinguish zeta-zero structure from easier controls.

## Next directions

A natural next notebook could do one of these:

1. compare against larger zero samples  
2. add pairwise spacing statistics beyond nearest neighbors  
3. examine local windows of \(|\zeta(1/2+it)|\) around each zero  
4. compare with a simple GUE-inspired synthetic control